In [ ]:

from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import threading
import datetime
import sqlite3
import threading
import pickle

DB_FILE = "log.db"

# Создаём таблицу один раз при запуске.
connection = sqlite3.connect(DB_FILE)

connection.execute("""
    CREATE TABLE IF NOT EXISTS logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        event_type TEXT NOT NULL,
        timestamp TEXT NOT NULL
    )
""")

columns = [
    row[1]
    for row in connection.execute("PRAGMA table_info(logs)")
]

# Добавляем длительность, если таблица была создана раньше без неё.
if "duration" not in columns:
    connection.execute(
        "ALTER TABLE logs ADD COLUMN duration REAL"
    )

connection.commit() 
connection.close()  


# Запись события в базу.
def add_log(event_type, timestamp=None, duration=None):
    if timestamp is None:
        timestamp = datetime.datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    connection = sqlite3.connect(DB_FILE)

    connection.execute(
        """
        INSERT INTO logs (event_type, timestamp, duration)
        VALUES (?, ?, ?)
        """,
        (event_type, timestamp, duration)
    )

    connection.commit()
    connection.close()

    return True


# Получение журнала с такими же фильтрами, как раньше.
def get_log(event_type=None,
            time_from=None, time_to=None,
            duration_from=None, duration_to=None):

    query = """
        SELECT timestamp, event_type, duration
        FROM logs
        WHERE 1=1
    """
    params = []

    if event_type is not None:
        query += " AND event_type = ?"
        params.append(event_type)

    if time_from is not None:
        query += " AND timestamp >= ?"
        params.append(time_from)

    if time_to is not None:
        query += " AND timestamp <= ?"
        params.append(time_to)

    if duration_from is not None:
        query += " AND duration >= ?"
        params.append(duration_from)

    if duration_to is not None:
        query += " AND duration <= ?"
        params.append(duration_to)

    query += " ORDER BY timestamp, id"

    connection = sqlite3.connect(DB_FILE)
    rows = connection.execute(query, params).fetchall()
    connection.close()

    if not rows:
        return "No records"

    lines = []

    for timestamp, event_type, duration in rows:
        if duration is None:
            duration_text = "не измерена"
        else:
            duration_text = f"{duration:.6f} сек."

        lines.append(
            f"{timestamp} | {event_type} | {duration_text}"
        )

    return "\n".join(lines)


stats_server = SimpleXMLRPCServer(
    ("127.0.0.1", 8028),
    allow_none=True,
    logRequests=False
)

stats_server.register_function(add_log, "add_log")
stats_server.register_function(get_log, "get_log")





print("Прокси запущен на порту 8009")
stats_server.serve_forever()


Прокси запущен на порту 8009
